In [ ]:
!pip install -q ultralytics
import yaml, shutil, random, pandas as pd, numpy as np
from pathlib import Path
from ultralytics import YOLO
inp, W = Path("/kaggle/input"), Path("/kaggle/working")

models = {p.parent.parent.name: p for p in inp.rglob("best.pt")}
print("Models:", sorted(models))
tz = next(p for p in inp.rglob("tz_split") if p.is_dir())
gh = next(p for p in inp.rglob("gh_split") if p.is_dir())

tile2photo = pd.read_csv(next(inp.rglob("tz_labels.csv")), dtype={"image": str}).drop_duplicates("image").set_index("image").photo
units = {"tz": {}, "gh": {}}
for p in (tz / "train" / "images").glob("*.jpg"):
    units["tz"].setdefault(tile2photo[p.stem.removeprefix("tz_")], []).append(p)
for p in (gh / "train" / "images").glob("*.jpg"):
    units["gh"].setdefault(p.stem.split("__")[0], []).append(p)
print("Tanzania train photos:", len(units["tz"]), "| Ghana train images:", len(units["gh"]))

for site, d in [("tz", tz), ("gh", gh)]:               
    shutil.copytree(d / "test", W / f"{site}_test" / "test", dirs_exist_ok=True)
    yaml.safe_dump({"path": str(W / f"{site}_test"), "train": "test/images", "val": "test/images",
                    "test": "test/images", "names": {0: "parasite", 1: "wbc"}}, open(W / f"{site}_test.yaml", "w"))

In [ ]:
def make_subset(site, k, seed):
    out = W / f"sub_{site}_{k}_{seed}"
    keys = sorted(units[site]); random.Random(seed).shuffle(keys)    
    for sub in ["images", "labels"]: (out / "train" / sub).mkdir(parents=True, exist_ok=True)
    for key in keys[:k]:
        for img in units[site][key]:
            shutil.copy(img, out / "train" / "images" / img.name)
            shutil.copy(img.parent.parent / "labels" / f"{img.stem}.txt", out / "train" / "labels" / f"{img.stem}.txt")
    y = W / f"sub_{site}_{k}_{seed}.yaml"
    yaml.safe_dump({"train": str(out / "train" / "images"), "val": str(out / "train" / "images"),
                    "names": {0: "parasite", 1: "wbc"}}, open(y, "w"))
    return y

def score(weights, site):
    m = YOLO(str(weights)).val(data=str(W / f"{site}_test.yaml"), split="test", imgsz=640, batch=32,
                               device=0, plots=False, verbose=False)
    return m.box.ap50[0], m.box.r[0], m.box.ap50[1]

rows = []
for site, base in [("tz", "F_bd_gh"), ("gh", "E_bd_tz")]:
    p, r, w = score(models[base], site)
    rows.append(dict(site=site, k=0, seed=0, par_mAP50=p, par_recall=r, wbc_mAP50=w))
    for seed in [0, 1]:
        for k in [25, 50, 100, 200]:
            name = f"ft_{site}_{k}_{seed}"
            try:
                YOLO(str(models[base])).train(data=str(make_subset(site, k, seed)), epochs=30, imgsz=640, batch=16,
                                              device=0, seed=seed, val=False, plots=False,
                                              project=str(W / "runs"), name=name, exist_ok=True)
                p, r, w = score(W / "runs" / name / "weights" / "last.pt", site)
                rows.append(dict(site=site, k=k, seed=seed, par_mAP50=p, par_recall=r, wbc_mAP50=w))
            except Exception as e:
                print("FAILED", name, e)
            pd.DataFrame(rows).to_csv(W / "few_shot_results.csv", index=False)  

res = pd.DataFrame(rows)
print(res.groupby(["site", "k"])[["par_mAP50", "par_recall", "wbc_mAP50"]].agg(["mean", "min", "max"]).round(3))